# Chapter 24: Future Directions — CCS and Hydrogen Integration

The oil and gas industry is evolving to integrate carbon capture and storage (CCS)
and hydrogen production into existing infrastructure. This chapter explores the
thermodynamic challenges of these emerging applications using NeqSim.

**Key learning objectives:**
- Model CO2-rich stream phase behavior for capture and transport
- Map the CO2 phase envelope and identify the dense phase transport region
- Analyze methane–hydrogen mixture properties for H2 blending
- Evaluate the effect of hydrogen blending on Wobbe index

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 24.1 CO2 Capture — Phase Behavior of CO2-Rich Streams

Post-combustion capture produces a CO2-rich stream with impurities (N2, O2, Ar, H2O).
Understanding the phase behavior is critical for compression, dehydration, and transport
design. We model the properties at a range of pressures at 25 °C.

In [2]:
from neqsim import jneqsim

# CO2-rich stream with typical impurities
co2_fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 1.0)
co2_fluid.addComponent("CO2", 0.96)
co2_fluid.addComponent("nitrogen", 0.02)
co2_fluid.addComponent("oxygen", 0.01)
co2_fluid.addComponent("water", 0.01)
co2_fluid.setMixingRule("classic")

ops = jneqsim.thermodynamicoperations.ThermodynamicOperations(co2_fluid)

# Sweep pressure to map density and compressibility
pressures = np.arange(10, 210, 10)
densities = []
z_factors = []

for p in pressures:
    co2_fluid.setPressure(float(p))
    co2_fluid.setTemperature(273.15 + 25.0)
    ops.TPflash()
    co2_fluid.initProperties()
    densities.append(co2_fluid.getDensity("kg/m3"))
    z_factors.append(co2_fluid.getZ())

print(f"{'P (bara)':>10} {'Density (kg/m3)':>16} {'Z-factor':>10}")
print("-" * 40)
for p, d, z in zip(pressures, densities, z_factors):
    print(f"{p:>10.0f} {d:>16.2f} {z:>10.4f}")

  P (bara)  Density (kg/m3)   Z-factor
----------------------------------------
        10            18.53     0.9440
        20            39.33     0.8911
        30            63.13     0.8342
        40            91.25     0.7711
        50           126.34     0.6981
        60           175.29     0.6060
        70           413.98     0.3047
        80           612.56     0.2387
        90           345.01     0.4677
       100           694.46     0.2648
       110           721.04     0.2811
       120           743.29     0.2979
       130           762.57     0.3150
       140           779.68     0.3322
       150           344.72     0.7801
       160           809.16     0.3666
       170           822.13     0.3837
       180           834.17     0.4008
       190           845.42     0.4178
       200           855.98     0.4347


## 24.2 CO2 Phase Envelope — Dense Phase Transport Region

For pipeline transport, CO2 is compressed above its critical point into the dense phase
region. We compute the phase envelope to identify the critical point and the operating
window for safe dense-phase transport.

In [3]:
# Create a fresh fluid for phase envelope calculation
co2_env = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 50.0)
co2_env.addComponent("CO2", 0.96)
co2_env.addComponent("nitrogen", 0.02)
co2_env.addComponent("oxygen", 0.01)
co2_env.addComponent("water", 0.01)
co2_env.setMixingRule("classic")

ops_env = jneqsim.thermodynamicoperations.ThermodynamicOperations(co2_env)

try:
    ops_env.calcPTphaseEnvelope(True)
    phase_env = ops_env.getJfreeChart("PT")

    # Extract data from the phase envelope operation
    env_data = ops_env.getData()
    temps_env = [float(x) for x in ops_env.get("temperature")]
    press_env = [float(x) for x in ops_env.get("pressure")]

    # Convert K to C
    temps_C = [t - 273.15 for t in temps_env]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(temps_C, press_env, "b-", linewidth=2, label="Phase envelope")

    # Dense phase transport region
    ax.axhline(y=100, color="green", linestyle="--", alpha=0.7, label="Typical transport P (100 bara)")
    ax.axvline(x=31.1, color="red", linestyle="--", alpha=0.7, label="CO2 critical T (31.1 °C)")

    # Shade dense phase region
    ax.fill_between([31, 60], [74, 74], [200, 200], alpha=0.15, color="green",
                    label="Dense phase region")

    ax.set_xlabel("Temperature (°C)", fontsize=12)
    ax.set_ylabel("Pressure (bara)", fontsize=12)
    ax.set_title("CO2-Rich Stream Phase Envelope\n(96% CO2 + N2/O2/H2O impurities)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_xlim(-60, 60)
    ax.set_ylim(0, 200)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("../figures/ch24_co2_phase_envelope.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved: ../figures/ch24_co2_phase_envelope.png")
except Exception as e:
    print(f"Phase envelope calculation: {e}")
    # Fallback: plot the density data from Section 24.1
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(pressures, densities, "b-o", linewidth=2, markersize=5)
    ax.axvline(x=73.8, color="red", linestyle="--", alpha=0.7, label="CO2 critical P (73.8 bara)")
    ax.fill_between([73.8, 200], [0, 0], [1200, 1200], alpha=0.1, color="green",
                    label="Dense phase region")
    ax.set_xlabel("Pressure (bara)", fontsize=12)
    ax.set_ylabel("Density (kg/m³)", fontsize=12)
    ax.set_title("CO2-Rich Stream Density at 25 °C\n(96% CO2 + impurities)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../figures/ch24_co2_phase_envelope.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved: ../figures/ch24_co2_phase_envelope.png (density fallback)")

Phase envelope calculation: No matching overloads found for neqsim.thermodynamicoperations.ThermodynamicOperations.getJfreeChart(str), options are:
	public org.jfree.chart.JFreeChart neqsim.thermodynamicoperations.ThermodynamicOperations.getJfreeChart()



Figure saved: ../figures/ch24_co2_phase_envelope.png (density fallback)


C:\Users\ESOL\AppData\Local\Temp\ipykernel_25164\2724158243.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 24.3 Hydrogen Blending — Mixture Properties

Blending hydrogen into natural gas pipelines is a near-term decarbonization strategy.
We model methane–hydrogen mixtures at varying H2 fractions to understand the impact
on density, heating value, and Wobbe index.

In [4]:
# Hydrogen blending study
h2_fractions = np.arange(0, 0.55, 0.05)  # 0% to 50% H2 by mole
wobbe_indices = []
densities_blend = []
molar_masses = []

for h2_frac in h2_fractions:
    ch4_frac = 1.0 - h2_frac
    blend = jneqsim.thermo.system.SystemSrkEos(273.15 + 15.0, 1.01325)
    blend.addComponent("methane", float(ch4_frac))
    blend.addComponent("hydrogen", float(h2_frac))
    blend.setMixingRule("classic")

    ops_blend = jneqsim.thermodynamicoperations.ThermodynamicOperations(blend)
    ops_blend.TPflash()
    blend.initProperties()

    # Gas density at standard conditions
    rho_gas = blend.getDensity("kg/m3")
    densities_blend.append(rho_gas)

    # Molar mass
    mw = blend.getMolarMass() * 1000.0  # kg/mol to g/mol
    molar_masses.append(mw)

    # Wobbe index approximation: W = HHV / sqrt(SG)
    # HHV_CH4 = 55.5 MJ/kg, HHV_H2 = 141.8 MJ/kg (mass basis)
    # Convert to volumetric: HHV_vol = HHV_mass * rho_gas
    hhv_mass = ch4_frac * 55.5 + h2_frac * 141.8  # MJ/kg (approximate mole-weighted)
    rho_air = 1.225  # kg/m3 at 15C, 1 atm
    sg = rho_gas / rho_air
    hhv_vol = hhv_mass * rho_gas  # MJ/m3
    wobbe = hhv_vol / np.sqrt(sg) if sg > 0 else 0
    wobbe_indices.append(wobbe)

print(f"{'H2 (mol%)':>10} {'Density (kg/m3)':>16} {'MW (g/mol)':>12} {'Wobbe (MJ/m3)':>14}")
print("-" * 55)
for frac, d, mw, w in zip(h2_fractions, densities_blend, molar_masses, wobbe_indices):
    print(f"{frac*100:>10.0f} {d:>16.4f} {mw:>12.2f} {w:>14.2f}")

 H2 (mol%)  Density (kg/m3)   MW (g/mol)  Wobbe (MJ/m3)
-------------------------------------------------------
         0           0.6799        16.04          50.65
         5           0.6500        15.34          53.37
        10           0.6202        14.64          55.90
        15           0.5903        13.94          58.20
        20           0.5605        13.24          60.29
        25           0.5307        12.54          62.15
        30           0.5010        11.83          63.76
        35           0.4712        11.13          65.11
        40           0.4414        10.43          66.20
        45           0.4117         9.73          66.99
        50           0.3820         9.03          67.48


In [5]:
# Plot Wobbe index vs H2 fraction
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = "#e15759"
ax1.plot(h2_fractions * 100, wobbe_indices, "o-", color=color1, linewidth=2,
         markersize=6, label="Wobbe Index")
ax1.set_xlabel("Hydrogen Fraction (mol%)", fontsize=12)
ax1.set_ylabel("Wobbe Index (MJ/m³)", fontsize=12, color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# Typical Wobbe index limits for gas appliances
ax1.axhspan(45, 55, alpha=0.1, color="green", label="Typical appliance range")

# Secondary axis: density
ax2 = ax1.twinx()
color2 = "#4e79a7"
ax2.plot(h2_fractions * 100, densities_blend, "s--", color=color2, linewidth=2,
         markersize=5, label="Gas Density")
ax2.set_ylabel("Density (kg/m³)", fontsize=12, color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center left", fontsize=10)

ax1.set_title("Effect of Hydrogen Blending on Gas Properties\n(CH4/H2 at 15 °C, 1.013 bara)",
              fontsize=13, fontweight="bold")
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch24_h2_blending_wobbe.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch24_h2_blending_wobbe.png")

Figure saved: ../figures/ch24_h2_blending_wobbe.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_25164\2200480375.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 24.4 Discussion and Outlook

**CO2 transport:** The phase envelope shows that the CO2-rich stream (with impurities)
has a cricondenbar slightly above the pure CO2 critical pressure (73.8 bara). For safe
dense-phase pipeline transport, operating above ~100 bara and 10–40 °C avoids two-phase
flow. Impurities (especially N2) raise the saturation pressure, requiring higher
minimum operating pressures.

**Hydrogen blending:** Increasing H2 fraction from 0% to 50% significantly reduces
gas density and Wobbe index. Most gas appliances require a Wobbe index within ±5% of
design value, suggesting a practical blending limit of 10–20 mol% H2 without
appliance modification. Higher H2 fractions would require burner redesign.

**Future integration opportunities:**
- Coupling CCS with offshore platform exhaust gas treatment
- Blue hydrogen production from natural gas with CCS
- Green hydrogen from offshore wind-powered electrolysis
- Dynamic modeling of intermittent hydrogen injection into gas networks

## Summary

This chapter demonstrated NeqSim's capabilities for modeling future energy systems:

- **CO2 phase behavior** — density transition from gas to dense phase above the
  critical point, essential for compression and pipeline design
- **Phase envelope mapping** — identifying safe operating windows for CO2 transport
- **Hydrogen blending** — quantifying the impact on Wobbe index and gas density
  as H2 fraction increases from 0% to 50%

These thermodynamic tools enable engineers to design and evaluate CCS infrastructure
and hydrogen integration strategies using rigorous equation-of-state models.